# Chapter 12 — Triton Layout Fundamentals

Part 5: Layouts and data movement.

**Thread.**

- **Recurring idea.** Every Triton tensor is two objects at once: a logical shape $(M,N)$, and a compiler-chosen physical map from each logical element to a lane and a register slot.
- **Chain of topics.** Shape → layout → distribution → elements/thread → warps → `tl.arange` → `tl.trans()` → layout conversion → distributed, blocked, and slice layouts → compatibility → performance.
- **Driving question.** For each tile: who owns each element, how many elements does each owner hold, and what does that map cost in memory transactions?

**Concept arc.**

- A Triton tensor has two lives: logically a rectangular array with shape $(M,N)$; physically scalar values spread over the lanes and registers of one program.
- The logical map decides **what** the kernel computes; the physical map decides **how fast** it runs.
- The chapter keeps the two separate and says explicitly when a statement is a teaching model rather than a compiler guarantee.

**Study tip.** Before touching Triton, draw the 8-element tile as a **2 threads × 4 elements** map:

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| thread | $T_0$ | $T_0$ | $T_0$ | $T_0$ | $T_1$ | $T_1$ | $T_1$ | $T_1$ |
| register | 0 | 1 | 2 | 3 | 0 | 1 | 2 | 3 |

- The picture has $2\cdot4=8$ slots, one per element, and teaches the two coordinates $(t,r)$ that every layout uses.
- On real hardware a warp has 32 lanes, so the same 8 elements usually land on 8 lanes with one element each (the worked example below).
- Use the worked example's lane map, not this paper map, when reasoning about memory coalescing.

**Learning objectives**

- Separate a tensor's logical shape from the compiler-chosen physical layout.
- Compute elements/thread, threads/program, and warps/program, and use those counts to predict register pressure and lane utilization.
- Write the canonical blocked map for a 1-D and a 2-D tile, and check a mapping table by hand.
- State what `tl.arange` produces and what `tl.trans` changes — and what it does not change.
- Recognize when a layout conversion is required and estimate its traffic.
- Predict coalescing efficiency from a thread-to-address map.


## Chapter front matter

- **Navigation.** [Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part4_broadcasting/ch11_index_transforms/11_complete.ipynb) · [Next →](../ch13_memory_optimization/13_complete.ipynb)
- **Time / difficulty.** 65–80 min · Intermediate.
- **Prerequisites.**
  - Chapter 11: reshape, transpose, permute, and slicing as logical operations.
  - Chapter 2: programs, grids, and `tl.program_id`.
  - Chapter 3: `tl.arange` and tensor shapes.
  - No prior GPU layout or shared-memory background is assumed.
- **Book apparatus.** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)
- **Scope.** This is not a description of Triton's internal IR. Triton chooses the concrete layout during lowering, depending on backend, shape, `num_warps`, and surrounding operations. The chapter uses one explicit **teaching model** so every table can be checked by hand.


## Concept diagrams

- **Logical tile vs physical distribution.** A logical tile is a rectangle of indices; a physical distribution assigns each index to one thread and one register slot.

```text
logical 4 x 4 tile                 one program, two warps (64 threads)

        col 0  col 1  col 2  col 3
row 0     0      1      2      3
row 1     4      5      6      7
row 2     8      9     10     11
row 3    12     13     14     15

teaching map (row-major flat index k = 4*i + j)
warp 0  owns k = 0..7    -> lanes 0..7, register 0
warp 1  owns k = 8..15   -> lanes 0..7, register 0
lanes 8..31 of both warps are idle
```

- **Blocked layout.** A blocked layout tiles the logical rectangle with a per-thread sub-block and then repeats that block over the tile.

```text
16 x 16 tile, sizePerThread=(2,2), threadsPerWarp=(4,8), warpsPerCTA=(1,1)

block coverage = (2*4*1) x (2*8*1) = 8 x 16
repeats        = ceil(16/8) x ceil(16/16) = 2 x 1
elements/thread = 2*2 per repeat * 2 repeats = 8

+---------------- 16 columns ----------------+
|  block 0: rows 0..7, cols 0..15  (8 x 16)  |
|  block 1: rows 8..15, cols 0..15 (8 x 16)  |
+--------------------------------------------+
```


- **Slice layout.** Slicing restricts the logical index set. Values keep their parent distribution, but the slice may be a non-contiguous window of the parent.

```text
parent 8 x 8                 slice rows 2..5, cols 1..4
+----------------+          +----------------+
|                |          |    +------+    |
|                |          |    | 4x4  |    |
|                |          |    +------+    |
+----------------+          +----------------+
16 of 64 elements = 25%
```

- **Layout conversion.** A conversion moves values between lanes or registers when a producer layout does not match what the consumer needs.

```text
producer layout A          consumer layout B
lane 0: a0 a1 a2 a3   ->   lane 0: a0 a4 a8 ...
lane 1: a4 a5 a6 a7   ->   lane 1: a1 a5 a9 ...
        ...                        ...
             \_____ convert_layout _____/
                (often via shared memory)
```

- **What stays fixed.** The conversion is a logical permutation; it does not change the tensor's shape or values, only which thread holds which value. It is not free.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)


## Part I — Concepts

**Thread.**

- Each card has the same shape: **Connection** (what it builds on), **Rule**, **Formula**, **Example**, **Where**, **Triton**, **Failure**, and **Bridge** (what comes next).
- The order is the dependency order: shape → layout → distribution → counts → `tl.arange` → `tl.trans` → conversion → distributed/blocked/slice → compatibility → performance.
- Read one card at a time; every table can be checked without a GPU.


### 12.1 Logical tensor shape

**Connection.** Builds on Chapter 11's logical transforms and Chapter 2's grid: before a layout can assign an element, the element needs a logical coordinate.

- **Rule.** A tensor's shape is the size of each logical axis; for a matrix tile the shape is $(M,N)$ and the number of elements is $MN$.
- **Formula.** $\operatorname{shape}(X)=(M,N)$ and $\operatorname{numel}(X)=MN$. With row-major order, $k=iN+j$, $i=\lfloor k/N\rfloor$, and $j=k\bmod N$.
- **Example.**

| Row $i$ | Columns | Flat indices $k$ |
|---:|---|---|
| 0 | 0,1,2,3 | 0,1,2,3 |
| 1 | 0,1,2,3 | 4,5,6,7 |
| 2 | 0,1,2,3 | 8,9,10,11 |
| 3 | 0,1,2,3 | 12,13,14,15 |

The table is a $4\times4$ tile: 16 logical elements, four rows, four columns.

- **Where.** Shape fixes the grid, the masks, and the pointer arithmetic; it does not fix which thread owns which element.
- **Triton.** Build the tile with two index vectors: `rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)`, `cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)`, and `offs = rows[:, None] * N + cols[None, :]`.
- **Failure.** Confusing shape with stride: `x.t()` has shape `(N,M)` but the same storage; `.contiguous()` changes strides, not logical values.
- **Bridge.** Section 12.2 adds the physical map on top of this logical shape.


### 12.2 Layout concepts

**Connection.** Builds on §12.1: the shape names the logical indices, and a layout assigns each one to a thread and register slot.

- **Rule.** A layout is a map from a logical index to a program-local thread and register slot; it must be injective on the live tile.
- **Formula.** $\mathcal L:(i,j)\mapsto(t,r)$ with $t\in\{0,\ldots,32W-1\}$ and $r\in\{0,\ldots,e_t-1\}$.
- **Example.**

| Logical flat index $k$ | Thread $t$ | Register $r$ | Comment |
|---:|---:|---:|---|
| 0–31 | $k$ | 0 | first value on each lane |
| 32–63 | $k-32$ | 1 | second value on each lane |

The table is a 64-element 1-D tile on one warp: 32 lanes, two elements per lane.

- **Where.** The layout decides which lane performs each scalar operation, which addresses are adjacent in a memory instruction, and when a value must move to another lane.
- **Triton.** You do not write $\mathcal L$; the compiler infers it from shapes, `num_warps`, and producer/consumer operations. Your levers are tile shape, launch shape, and operation order.
- **Failure.** Assuming a fixed rule such as "lane 0 always owns element 0"; that is true in the teaching model and false in general.
- **Bridge.** Section 12.3 makes the map concrete by partitioning the flat index set.


### 12.3 Element distribution

**Connection.** Builds on §12.2: a layout is the map, and this section chooses one explicit teaching map for every later table.

- **Rule.** Distribution partitions $[0,T)$ among $32W$ threads using a **warp-blocked, lane-interleaved** model; all examples divide evenly.
- **Formula.** $C=\lceil T/W\rceil$, $w=\lfloor k/C\rfloor$, $\ell=k-wC$, $\text{lane}=\ell\bmod 32$, $\text{register}=\lfloor\ell/32\rfloor$, and $t=32w+\text{lane}$.
- **Example.**

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| warp $w$ | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| lane | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| register | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| active? | yes | yes | yes | yes | yes | yes | yes | yes |

Lanes 8–31 are idle; every live $k$ has exactly one owner.

- **Where.** Adjacent $k$ on adjacent lanes gives coalesced memory access; a different distribution can turn one 128-byte transaction into 32.
- **Triton.** Distribution is compiler-chosen; influence it with tile shape and `num_warps`, and observe it through memory traffic, not a public "thread id" API.
- **Failure.** Forgetting that small tiles leave lanes idle, or assuming "adjacent logical index = adjacent lane" for every operation.
- **Bridge.** Section 12.4 turns the partition into the count that drives register pressure.


### 12.4 Elements per thread

**Connection.** Builds on §12.3: once the tile is partitioned, the quotient of tile volume and thread count is the per-thread workload.

- **Rule.** The average number of tile elements per thread is $T/(32W)$; the actual count is an integer, so the tile pads to $32W\lceil T/(32W)\rceil$.
- **Formula.** $\bar e=T/(32W)$, $T'=32W\lceil T/(32W)\rceil$, and register demand is roughly $R\,e_t$ for $R$ live tensors.
- **Example.**

| $T$ | $W$ | Threads $32W$ | $\bar e$ | Padded $T'$ | Active lanes |
|---:|---:|---:|---:|---:|---:|
| 8 | 1 | 32 | 0.25 | 32 | 8 |
| 16 | 2 | 64 | 0.25 | 64 | 16 |
| 1024 | 4 | 128 | 8 | 1024 | 128 |
| 4096 | 4 | 128 | 32 | 4096 | 128 |
| 4096 | 8 | 256 | 16 | 4096 | 256 |

Check the third row: $1024/(32\cdot4)=8$; the fourth: $4096/128=32$.

- **Where.** This is the main register-pressure knob: too many elements per thread spills to local memory, and too few leaves lanes idle.
- **Triton.** Choose `BLOCK_*` and `num_warps` together: 4096 elements with 4 warps is 32 values/thread, with 8 warps is 16.
- **Failure.** Increasing `BLOCK` without `num_warps` (spills), or increasing `num_warps` without growing the tile (idle lanes).
- **Bridge.** Section 12.5 fixes one factor in the denominator: the warp width.


### 12.5 Threads per warp

**Connection.** Builds on §12.4: the count $32W$ uses two hardware facts, and this section fixes the first, the warp width.

- **Rule.** On NVIDIA a warp is 32 threads; AMD wavefronts are 32 or 64 depending on the architecture, and this book uses 32 as the teaching value.
- **Formula.** $\text{threads per warp}=32$ and $\text{threads/program}=32W$.
- **Example.**

| $W$ (warps/program) | Threads/program on NVIDIA |
|---:|---:|
| 1 | 32 |
| 2 | 64 |
| 4 | 128 |
| 8 | 256 |

- **Where.** The warp is the scheduling and memory-coalescing unit: one instruction covers 32 lanes and one memory instruction touches their addresses together.
- **Triton.** `num_warps` is a launch/compile parameter; common values are powers of two.
- **Failure.** Treating a warp as 64 threads on NVIDIA, or confusing a Triton program with a CUDA thread.
- **Bridge.** Section 12.6 fixes the second factor, the number of warps in a program.


### 12.6 Warps per program

**Connection.** Builds on §12.5: the warp width is fixed at 32, so the program-wide thread count is the remaining free factor.

- **Rule.** A Triton program contains $W$ warps, and $P$ programs contain $P\cdot32W$ threads.
- **Formula.** $\text{threads/program}=32W$ and $\text{total threads}=P\cdot32W$.
- **Example.** A 64×64 tile split into $P=4$ programs gives 1024 elements per program.

| $W$ | Threads/program | Elements/thread | Total threads |
|---:|---:|---:|---:|
| 2 | 64 | 16 | 256 |
| 4 | 128 | 8 | 512 |
| 8 | 256 | 4 | 1024 |

- **Where.** More warps hide memory latency and raise occupancy, but divide the same tile into fewer elements per thread and add scheduling overhead.
- **Triton.** The grid is the launch tuple; `num_warps` is a launch keyword, both visible in the wrapper rather than the kernel body.
- **Failure.** Raising `num_warps` and expecting the same elements/thread; the tile did not grow, so work per thread fell.
- **Bridge.** Section 12.7 moves from program-level counts to the index vector that names elements inside a tile.


### 12.7 `tl.arange()` layouts

**Connection.** Builds on §12.6: with warps and threads counted, `tl.arange` is how a program manufactures the logical indices those threads will process.

- **Rule.** `tl.arange(start, end)` builds consecutive integers; its logical shape is $(B,)$ and its physical layout is compiler-chosen.
- **Formula.** $a_k=\text{start}+k$ for $k=0,\ldots,B-1$, with $B=\text{end}-\text{start}$.
- **Example.**

| $k$ | Warp | Lane | Register | Value |
|---:|---:|---:|---:|---:|
| 0–31 | 0 | $k$ | 0 | $k$ |
| 32–63 | 1 | $k-32$ | 0 | $k$ |

This is a $B=64$ index vector with $W=2$; for $B=8$ and $W=1$, only lanes 0–7 hold values.

- **Where.** `tl.arange` manufactures offsets, masks, row/column vectors, and gather addresses.
- **Triton.** Use a compile-time power-of-two length: `tl.arange(0, BLOCK)`; combine it with `[None, :]` or `[:, None]`.
- **Failure.** A non-power-of-two length, or treating an index vector as data; the values are integers but the layout is still a vector.
- **Bridge.** Section 12.8 adds the operation that changes a tile's logical axes.


### 12.8 `tl.trans()`

**Connection.** Builds on §12.7: index vectors build tiles in one logical order, and `tl.trans` is the operation that swaps that order.

- **Rule.** `tl.trans(x)` reverses the tensor's axes; for a matrix it swaps rows and columns.
- **Formula.** $y_{ji}=x_{ij}$ and $\operatorname{shape}(y)=(N,M)$.
- **Example.**

| $x$ (2×3) | | | $y$ (3×2) | |
|---|---|---|---|---|
| 0 | 1 | 2 | 0 | 3 |
| 3 | 4 | 5 | 1 | 4 |
| | | | 2 | 5 |

- **Where.** Attention ($K^\top$), matmul operand order, and any kernel that reads one order and writes another.
- **Triton.** `xt = tl.trans(x)` changes the logical view inside the kernel; it does not move global memory by itself, because the store decides where values land.
- **Failure.** Assuming `tl.trans` is free or identical to `torch.transpose` on a strided tensor; `torch.transpose` changes strides, while `tl.trans` changes a distributed tile and may force a conversion.
- **Bridge.** Section 12.9 explains that forced conversion.


### 12.9 Layout conversion

**Connection.** Builds on §12.8: a transpose can change the required layout, and this section describes the rewrite that reconciles producer and consumer.

- **Rule.** A conversion rewrites $\mathcal L$ while preserving logical values; if staged through shared memory it costs at least one write and one read per tile.
- **Formula.** $\mathcal L_{\text{out}}\neq\mathcal L_{\text{in}}$, $\operatorname{values}_{\text{out}}=\operatorname{values}_{\text{in}}$, and $Q_{\text{conv}}\ge2Tb$ bytes per tile.
- **Example.**

| Tile | $T$ | $b$ (fp32) | $2Tb$ per conversion | 1000 tiles |
|---|---:|---:|---:|---:|
| 32×32 | 1024 | 4 | 8192 B | 8,192,000 B |

- **Where.** Reductions, transposes, `tl.dot` operands, and elementwise combinations of tensors with different histories.
- **Triton.** Keep producers and consumers in the same layout; apply `tl.trans` once, not once per operation. The compiler inserts conversions automatically.
- **Failure.** Converting back and forth in a loop, or transposing only to combine with an untransposed tensor.
- **Bridge.** Section 12.10 zooms out from one program's tile to a tensor distributed over the grid.


### 12.10 Distributed tensors

**Connection.** Builds on §12.9: conversions happen inside a program, and distributed tensors explain how programs partition the global tensor in the first place.

- **Rule.** A Triton tensor is distributed over the threads of one program; a global tensor is distributed over the grid, with program $p$ owning tile $X_p$.
- **Formula.** $X=\bigcup_{p=0}^{P-1}X_p$, and inside a program $T=32W\,e_t$ when the tile divides evenly.
- **Example.**

| Global shape | $P$ | Tile per program | $W$ | Threads/program | $e_t$ | Total threads |
|---|---:|---|---:|---:|---:|---:|
| 64×64 | 4 | 32×32 | 4 | 128 | 8 | 512 |

Check: $64\cdot64=4096$ elements; each program owns $32\cdot32=1024$; with $128$ threads, $1024/128=8$ values per thread.

- **Where.** Every Triton operation is SPMD over distributed tensors; a reduction along a distributed axis must communicate across lanes.
- **Triton.** Think in per-program tiles; the kernel body has no loop over threads, because the compiler maps the tile.
- **Failure.** Writing code as if one thread owns the whole tile, or indexing a global tensor without the program offset.
- **Bridge.** Section 12.11 describes the regular per-thread tiling used by elementwise and copy kernels.


### 12.11 Blocked layouts

**Connection.** Builds on §12.10: a distributed tensor is partitioned across programs, and a blocked layout partitions each program's tile across threads.

- **Rule.** A blocked layout repeats a per-thread sub-block over the tile; per-axis coverage is $C_d=s_dt_dw_d$ and the repeat count is $R_d=\lceil B_d/C_d\rceil$.
- **Formula.** $s=(s_0,s_1)$, $t=(t_0,t_1)$ with $t_0t_1=32$, $w=(w_0,w_1)$, $C_d=s_dt_dw_d$, and $R_d=\lceil B_d/C_d\rceil$.
- **Example.**

| Parameter | Value | Meaning |
|---|---|---|
| tile $B$ | 16×16 | logical rectangle |
| $s$ | (2,2) | 2×2 values per thread per block |
| $t$ | (4,8) | 4×8 = 32 threads per warp |
| $w$ | (1,1) | one warp per program |
| coverage $C$ | 8×16 | one block covers 8 rows, 16 columns |
| repeats $R$ | 2×1 | two row repeats |
| elements/thread | 8 | $2\cdot2\cdot2$ |

Check: $16\cdot16=256$ elements over 32 threads is 8 per thread.

- **Where.** Blocked is the mental model for elementwise and copy kernels; keep the contiguous dimension last and $s$ small there for coalescing.
- **Triton.** You do not set $s,t,w$ directly; choose tile shape, offset order, and `num_warps`. Lowercase $w$ is the per-axis tuple; scalar $W$ is `num_warps`.
- **Failure.** Assuming every layout is blocked; dot operands, reductions, and transposes use specialized layouts, which is where conversions appear.
- **Bridge.** Section 12.12 restricts a blocked tile to a sub-rectangle.


### 12.12 Slice layouts

**Connection.** Builds on §12.11: a slice is a restricted window of the blocked parent tile.

- **Rule.** A slice restricts the logical index set of a parent; values keep the parent distribution, but the window may be non-contiguous.
- **Formula.** $Y_{ij}=X_{i+s_m,j+s_n}$ for $0\le i\lt H$ and $0\le j\lt K$.
- **Example.**

| Parent | Slice origin | Slice shape | Elements | Fraction |
|---|---|---|---:|---:|
| 8×8 | $(s_m,s_n)=(2,1)$ | 4×4 | 16 | 0.25 |

The corners map as $(0,0)_Y\mapsto(2,1)_X$ and $(3,3)_Y\mapsto(5,4)_X$.

- **Where.** Attention head slices, KV-cache windows, boundary tiles, and any sub-rectangle processing.
- **Triton.** Prefer explicit offset vectors over Python slicing; mask with parent coordinates, not slice-local ones.
- **Failure.** Assuming a slice is contiguous; a row slice is contiguous only when it spans whole rows, and a column slice of a wider row-major matrix is not.
- **Bridge.** Section 12.13 asks when two sliced or otherwise produced tensors can be combined.


### 12.13 Layout compatibility

**Connection.** Builds on §12.12: slices and transposes produce tensors with different histories, and compatibility decides whether they can meet elementwise.

- **Rule.** Two tensors combine elementwise when their shapes broadcast and their layouts can be reconciled; otherwise the compiler inserts a conversion or rejects the shapes.
- **Formula.** $\operatorname{shape}(a)\sim\operatorname{shape}(b)$ and there exists a common $\mathcal L$ for both operands.
- **Example.**

| Operation | Shapes | Layouts | Result |
|---|---|---|---|
| `x + y` | same | same | no conversion |
| `tl.trans(x) + y` | same after transpose | different | conversion inserted |
| `x + b` | `(M,N)` and `(N,)` | broadcast | broadcast, possibly free |
| `x + y` | `(M,N)` and `(M+1,N)` | — | shape error |

- **Where.** Compatibility explains why two kernels with identical arithmetic can differ in speed: one keeps one layout, the other converts.
- **Triton.** Combine tensors with shared provenance; if you transpose, do the elementwise work in the transposed layout or convert once deliberately.
- **Failure.** Assuming "same shape" means "same layout"; shape compatibility is necessary, not sufficient.
- **Bridge.** Section 12.14 turns compatibility and distribution into a bandwidth model.


### 12.14 Layout performance

**Connection.** Builds on §12.13: compatible layouts avoid conversions, and this section prices the layouts that remain.

- **Rule.** Layout affects time through effective traffic; efficiency is useful bytes over moved bytes.
- **Formula.** $t\ge Q_{\text{eff}}/\mathrm{BW}$, $\eta=Q_{\text{useful}}/Q_{\text{eff}}$, and for a stride-$s$ warp read with 128-byte lines and no reuse, $\text{transactions}=\min(32,\max(1,s))$ and $Q_{\text{eff}}=128\cdot\text{transactions}$.
- **Example.**

| Stride $s$ | Transactions | Bytes moved | Useful bytes | $\eta$ |
|---:|---:|---:|---:|---:|
| 1 | 1 | 128 | 128 | 1 |
| 2 | 2 | 256 | 128 | 0.5 |
| 4 | 4 | 512 | 128 | 0.25 |
| 8 | 8 | 1024 | 128 | 0.125 |
| 16 | 16 | 2048 | 128 | 0.0625 |
| 32 | 32 | 4096 | 128 | 0.03125 |
| 4096 | 32 | 4096 | 128 | 0.03125 |

- **Where.** Memory-bound kernels are dominated by bytes, not FLOPs; a bad layout can waste 2–32× bandwidth before arithmetic runs.
- **Triton.** Keep the contiguous dimension innermost, align tile bases, and choose `num_warps` so elements/thread stays moderate; measure, because the model predicts direction, not exact time.
- **Failure.** Optimizing arithmetic while the layout issues 32 transactions for 128 useful bytes.
- **Bridge.** Part II implements this model on the host so the tables can be checked before any kernel runs.


### What the compiler owns, and what you own

| Decision | Owner | Lever |
|---|---|---|
| concrete layout $\mathcal L$ | compiler | shape, `num_warps`, producer/consumer ops |
| tile shape | you | `BLOCK_*` constants |
| grid | you | launch tuple |
| elements/thread | you, indirectly | tile volume / (32·`num_warps`) |
| conversion insertion | compiler | operation order, `tl.trans` placement |
| coalescing quality | you, indirectly | offset order and stride pattern |

- **Boundary.** Tables describe the logical model, kernel tests verify logical semantics, and no cell claims to read a register assignment out of the compiler.
- **Where this leads.** The chapter map and worked example turn these rules into one hand-checkable map; Parts II–IV then implement that map on the host and in Triton.


## Chapter map

- **How to read the table.** Each row names the formula or rule, the layout implication, and the runnable demo.

| § | Topic | Formula/rule | Layout implication | Kernel/demo |
|---|---|---|---|---|
| **12.1** | Logical tensor shape | $\operatorname{shape}=(M,N)$, $k=iN+j$ | shape fixes grid and mask, not owner | 2-D tile copy |
| **12.2** | Layout concepts | $\mathcal L:(i,j)\mapsto(t,r)$ | owner and memory adjacency | host map |
| **12.3** | Element distribution | $w=\lfloor k/C\rfloor,\ \ell=k-wC$ | adjacent lanes → coalescing | 8-element table |
| **12.4** | Elements per thread | $\bar e=T/(32W)$ | register pressure / idle lanes | count table |
| **12.5** | Threads per warp | 32 on NVIDIA | scheduling and transaction unit | `num_warps` sweep |
| **12.6** | Warps per program | $32W$ threads/program | latency hiding vs work per thread | `num_warps` sweep |
| **12.7** | `tl.arange` layouts | $a_k=\text{start}+k$ | index-vector layout | arange kernel |
| **12.8** | `tl.trans()` | $y_{ji}=x_{ij}$ | swaps axes, may convert | trans kernel |
| **12.9** | Layout conversion | $Q_{\text{conv}}\ge2Tb$ | shared-memory round trip | tiled transpose |
| **12.10** | Distributed tensors | $X=\bigcup_p X_p$ | per-program ownership | tiled transpose |
| **12.11** | Blocked layouts | $C_d=s_dt_dw_d$ | regular per-thread sub-blocks | 16×16 blocked table |
| **12.12** | Slice layouts | $Y_{ij}=X_{i+s_m,j+s_n}$ | non-contiguous sub-window | offset tile |
| **12.13** | Layout compatibility | same shape + common layout | conversions at elementwise joins | trans + add |
| **12.14** | Layout performance | $t\ge Q_{\text{eff}}/\mathrm{BW}$ | $1/s$ coalescing efficiency | traffic model |


## Worked numerical example

- **Configuration.** One program with $W$ warps of 32 lanes; the map is the warp-blocked, lane-interleaved model of §12.3.

### The 8-element 1-D tile

- **Setup.** $T=8$, $W=1$: one warp, 32 lanes, 8 live elements.
- **Map.**

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| thread $t$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| warp | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| lane | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| register | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |

- **Counts.** Average $\bar e=8/32=0.25$; one value on 8 active lanes; utilization $8/32=25\%$.

### The 4×4 2-D tile over two warps

- **Setup.** Flatten row-major, $k=4i+j$, with $T=16$ and $W=2$; the warp chunk is $C=\lceil16/2\rceil=8$.
- **Map.**

| $(i,j)$ | $k$ | thread | warp | lane | register |
|---|---:|---:|---:|---:|---:|
| (0,0) | 0 | 0 | 0 | 0 | 0 |
| (0,1) | 1 | 1 | 0 | 1 | 0 |
| (0,2) | 2 | 2 | 0 | 2 | 0 |
| (0,3) | 3 | 3 | 0 | 3 | 0 |
| (1,0) | 4 | 4 | 0 | 4 | 0 |
| (1,1) | 5 | 5 | 0 | 5 | 0 |
| (1,2) | 6 | 6 | 0 | 6 | 0 |
| (1,3) | 7 | 7 | 0 | 7 | 0 |
| (2,0) | 8 | 32 | 1 | 0 | 0 |
| (2,1) | 9 | 33 | 1 | 1 | 0 |
| (2,2) | 10 | 34 | 1 | 2 | 0 |
| (2,3) | 11 | 35 | 1 | 3 | 0 |
| (3,0) | 12 | 36 | 1 | 4 | 0 |
| (3,1) | 13 | 37 | 1 | 5 | 0 |
| (3,2) | 14 | 38 | 1 | 6 | 0 |
| (3,3) | 15 | 39 | 1 | 7 | 0 |

- **Counts.** Average $\bar e=16/64=0.25$; one value on 16 active lanes; utilization $16/64=25\%$.
- **Reading.** Every live element is in register 0; 48 of the 64 threads hold nothing.

### Where a second register appears

- **8×8 tile.** $T=64$ on one warp gives $e_t=2$.

| $k$ | Warp | Lane | Register |
|---:|---:|---:|---:|
| 0–31 | 0 | $k$ | 0 |
| 32–63 | 0 | $k-32$ | 1 |

- **64×64 tile.** $T=4096$; 4 warps gives $T/(32\cdot4)=32$ values/thread, and 8 warps gives 16.
- **Check.** The next code cell asserts every number in these tables.


In [ ]:
def cdiv(a, b):
    """Ceiling division for positive integers."""
    return -(-a // b)


def teaching_layout(k, total, warps):
    """Teaching model: warp-blocked, lane-interleaved flat index map."""
    chunk = cdiv(total, warps)
    warp = min(k // chunk, warps - 1)
    local = k - warp * chunk
    lane = local % 32
    register = local // 32
    return 32 * warp + lane, warp, lane, register


def test_worked_tables():
    # 8-element tile on one warp.
    assert teaching_layout(0, 8, 1) == (0, 0, 0, 0)
    assert teaching_layout(7, 8, 1) == (7, 0, 7, 0)
    assert 8 / (32 * 1) == 0.25
    assert 32 * 1 * cdiv(8, 32) == 32

    # 4x4 tile over two warps.
    assert teaching_layout(0, 16, 2) == (0, 0, 0, 0)
    assert teaching_layout(7, 16, 2) == (7, 0, 7, 0)
    assert teaching_layout(8, 16, 2) == (32, 1, 0, 0)
    assert teaching_layout(15, 16, 2) == (39, 1, 7, 0)
    assert 16 / (32 * 2) == 0.25
    assert 32 * 2 * cdiv(16, 64) == 64

    table = []
    for i in range(4):
        for j in range(4):
            k = 4 * i + j
            t, w, lane, reg = teaching_layout(k, 16, 2)
            table.append((i, j, k, t, w, lane, reg))
    assert table[0] == (0, 0, 0, 0, 0, 0, 0)
    assert table[8] == (2, 0, 8, 32, 1, 0, 0)
    assert table[15] == (3, 3, 15, 39, 1, 7, 0)
    assert len({row[3] for row in table}) == 16

    # 8x8 tile on one warp: two registers per lane.
    assert teaching_layout(31, 64, 1) == (31, 0, 31, 0)
    assert teaching_layout(32, 64, 1) == (0, 0, 0, 1)
    assert teaching_layout(63, 64, 1) == (31, 0, 31, 1)
    assert 64 / (32 * 1) == 2

    # 64x64 tile: 4096 elements, 4 and 8 warps.
    assert 4096 / (32 * 4) == 32
    assert 4096 / (32 * 8) == 16
    assert 256 / (32 * 1) == 8


test_worked_tables()
print("worked tables: 8-element, 4x4, and register-slot maps checked")


## Part II — Method 01: a host-side layout simulation

- **What it is.** A host function that takes a logical flat index and returns the thread, warp, lane, and register predicted by the teaching model.
- **Why.** It checks every table in Part I and builds the inverse map without a GPU.
- **Scope.** It is the reference for the logical rules, not a model of the compiler; if the compiler chooses another layout, values are unchanged and only the performance model changes.


In [ ]:
def row_major_index(i, j, n):
    """Row-major flat index of logical coordinate (i, j)."""
    return i * n + j


def owner_of(i, j, shape, warps):
    """Owner of a 2-D logical coordinate under the teaching model."""
    m, n = shape
    k = row_major_index(i, j, n)
    return teaching_layout(k, m * n, warps)


def inverse_layout(thread, register, total, warps):
    """Recover the flat index from (thread, register), or None if padded."""
    warp = thread // 32
    lane = thread % 32
    chunk = cdiv(total, warps)
    local = register * 32 + lane
    k = warp * chunk + local
    return k if 0 <= k < total else None


def test_host_layout_map():
    assert owner_of(0, 0, (4, 4), 2) == (0, 0, 0, 0)
    assert owner_of(2, 1, (4, 4), 2) == (33, 1, 1, 0)
    assert owner_of(3, 3, (4, 4), 2) == (39, 1, 7, 0)
    assert owner_of(0, 0, (8, 8), 1) == (0, 0, 0, 0)
    assert owner_of(4, 0, (8, 8), 1) == (0, 0, 0, 1)


def test_inverse_layout():
    for total, warps in ((8, 1), (16, 2), (64, 1), (1024, 4)):
        seen = set()
        for k in range(total):
            thread, warp, lane, register = teaching_layout(k, total, warps)
            assert 0 <= thread < 32 * warps
            assert 0 <= lane < 32
            assert inverse_layout(thread, register, total, warps) == k
            seen.add(thread)
        assert len(seen) == min(total, 32 * warps)


test_host_layout_map()
test_inverse_layout()
print("host simulation: forward map and inverse agree on four tiles")


### Table arithmetic

- **What the cell checks.** Counts from §12.4, §12.5, §12.6, §12.9, §12.10, §12.11, §12.12, and §12.14.
- **How.** Pure integer and float arithmetic, silent asserts, no table printing.


In [ ]:
def test_layout_tables():
    # 12.1: 4x4 tile and row-major inverse.
    assert 4 * 4 == 16
    for i in range(4):
        for j in range(4):
            k = 4 * i + j
            assert k // 4 == i and k % 4 == j

    # 12.2: 64 elements on one warp -> 2 registers per lane.
    for k in range(64):
        thread, warp, lane, register = teaching_layout(k, 64, 1)
        assert thread == k % 32
        assert register == k // 32

    # 12.4: elements/thread and padding.
    table_12_4 = [
        (8, 1, 0.25, 32, 8),
        (16, 2, 0.25, 64, 16),
        (1024, 4, 8.0, 1024, 128),
        (4096, 4, 32.0, 4096, 128),
        (4096, 8, 16.0, 4096, 256),
    ]
    for total, warps, average, padded, active in table_12_4:
        assert total / (32 * warps) == average
        assert 32 * warps * cdiv(total, 32 * warps) == padded
        assert min(total, 32 * warps) == active

    # 12.5: threads per program.
    assert [32 * w for w in (1, 2, 4, 8)] == [32, 64, 128, 256]

    # 12.6: 1024 elements per program.
    for warps, per_thread, total_threads in ((2, 16.0, 256), (4, 8.0, 512), (8, 4.0, 1024)):
        assert 1024 / (32 * warps) == per_thread
        assert 4 * 32 * warps == total_threads

    # 12.7: arange B=64 with two warps.
    assert teaching_layout(0, 64, 2)[1:] == (0, 0, 0)
    assert teaching_layout(31, 64, 2)[1:] == (0, 31, 0)
    assert teaching_layout(32, 64, 2)[1:] == (1, 0, 0)
    assert teaching_layout(63, 64, 2)[1:] == (1, 31, 0)

    # 12.8: logical transpose values.
    x = torch.tensor([[0, 1, 2], [3, 4, 5]])
    assert torch.equal(x.t(), torch.tensor([[0, 3], [1, 4], [2, 5]]))

    # 12.9: one 32x32 fp32 conversion.
    assert 32 * 32 == 1024
    assert 2 * 1024 * 4 == 8192
    assert 1000 * 8192 == 8_192_000

    # 12.10: distributed 64x64 over four programs.
    assert 64 * 64 == 4096
    assert (64 * 64) // 4 == 1024
    assert 1024 // (32 * 4) == 8
    assert 4 * 128 == 512

    # 12.11: blocked 16x16, s=(2,2), t=(4,8), w=(1,1).
    s = (2, 2)
    t = (4, 8)
    w = (1, 1)
    coverage = (s[0] * t[0] * w[0], s[1] * t[1] * w[1])
    repeats = (cdiv(16, coverage[0]), cdiv(16, coverage[1]))
    assert t[0] * t[1] == 32
    assert coverage == (8, 16)
    assert repeats == (2, 1)
    assert s[0] * s[1] * repeats[0] * repeats[1] == 8
    assert 16 * 16 // 32 == 8

    # 12.12: an 8x8 parent sliced at (2,1) to 4x4.
    assert 8 * 8 == 64
    assert 4 * 4 == 16
    assert 16 / 64 == 0.25
    assert (0 + 2, 0 + 1) == (2, 1)
    assert (3 + 2, 3 + 1) == (5, 4)

    # 12.13: broadcast compatibility is shape arithmetic.
    assert (torch.zeros(4, 3) + torch.zeros(4, 1)).shape == (4, 3)
    assert (torch.zeros(4, 3) + torch.zeros(3)).shape == (4, 3)

    # 12.14: stride table.
    for stride in (1, 2, 4, 8, 16, 32, 4096):
        transactions = min(32, max(1, stride))
        moved = 128 * transactions
        useful = 32 * 4
        assert moved == 128 * transactions
        assert useful / moved == 1 / transactions


test_layout_tables()


**What the simulation proves.**

- **Injective.** The map sends every live $k$ to one owner, and the inverse recovers the original index.
- **Utilization.** An 8-element tile on a 32-lane warp is 25% occupied; a 4×4 tile on two warps is 25% occupied.
- **Limit.** It does not prove the compiler uses this map.


## Part III — Method 02: kernels that expose layout behavior

- **Kernel 1.** `tl.arange` builds a 1-D index vector.
- **Kernel 2.** `[:, None]` and `[None, :]` build a 2-D tile from two 1-D index vectors.
- **Kernel 3.** `tl.trans` changes the logical axes of a distributed tile.
- **Verification.** Each test compares against PyTorch and asserts values and shapes; none inspects registers, because the concrete layout is compiler-owned.


In [ ]:
@triton.jit
def arange_kernel(out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    tl.store(out_ptr + offs, offs, mask=mask)


def test_arange_kernel():
    n, block = 13, 16
    out = torch.empty(n, dtype=torch.int32)
    arange_kernel[(1,)](out, n, BLOCK=block)
    torch.testing.assert_close(out, torch.arange(n, dtype=torch.int32))


if HAS_GPU:
    test_arange_kernel()
    print("tl.arange: 13 live lanes of a 16-lane block")


In [ ]:
@triton.jit
def tile2d_kernel(x_ptr, out_ptr, M, N,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    offs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, 2.0 * x, mask=mask)


def test_tile2d_kernel():
    M, N = 5, 7
    x = torch.arange(M * N, dtype=torch.float32).view(M, N)
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, 8), triton.cdiv(N, 8))
    tile2d_kernel[grid](x, out, M, N, BLOCK_M=8, BLOCK_N=8)
    torch.testing.assert_close(out, 2.0 * x)


if HAS_GPU:
    test_tile2d_kernel()
    print("2-D tile: rows[:,None] * N + cols[None,:] covers a 5x7 tile")


In [ ]:
@triton.jit
def trans_kernel(x_ptr, out_ptr, M, N,
                 BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    rows = tl.arange(0, BLOCK_M)
    cols = tl.arange(0, BLOCK_N)
    load_mask = (rows[:, None] < M) & (cols[None, :] < N)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :],
                mask=load_mask, other=0.0)
    xt = tl.trans(x)
    store_mask = (cols[:, None] < N) & (rows[None, :] < M)
    tl.store(out_ptr + cols[:, None] * M + rows[None, :], xt, mask=store_mask)


def test_trans_kernel():
    M, N = 3, 5
    x = torch.arange(M * N, dtype=torch.float32).view(M, N)
    out = torch.empty(N, M, dtype=torch.float32)
    trans_kernel[(1,)](x, out, M, N, BLOCK_M=4, BLOCK_N=8)
    torch.testing.assert_close(out, x.t())


if HAS_GPU:
    test_trans_kernel()
    print("tl.trans: (3,5) tile becomes (5,3); values match x.t()")


**Layout warning.**

- **Single-tile blocks.** The kernels above use them so the logical map is easy to read.
- **Lowering.** `tl.trans` may compile to a register permutation or a shared-memory round trip, depending on backend and consumer.
- **What is checked.** Only values are asserted; Part IV shows why conversion is not free and how a tiled transpose restores coalescing.


## Part IV — Method 03: a transpose kernel and the cost of a layout change

- **Read vs write order.** A transpose reads one order and writes another.
- **Coalesced store.** Loading a row-major tile and storing it row-major in the output keeps the store coalesced; the price is a layout conversion inside the program.
- **Strided store.** Storing in input order without converting makes the store strided, and the memory system pays instead.
- **Tiling.** The kernel below converts explicitly with `tl.trans`. The grid arithmetic is $P_m=\lceil M/B_M\rceil$, $P_n=\lceil N/B_N\rceil$, and $P=P_mP_n$; for $M=40$, $N=24$, $B_M=16$, $B_N=8$, this gives $P_m=3$, $P_n=3$, and $P=9$.


In [ ]:
@triton.jit
def transpose_tiled_kernel(x_ptr, out_ptr, M, N,
                           BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    load_mask = (rows[:, None] < M) & (cols[None, :] < N)
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :],
                mask=load_mask, other=0.0)
    xt = tl.trans(x)
    store_mask = (cols[:, None] < N) & (rows[None, :] < M)
    tl.store(out_ptr + cols[:, None] * M + rows[None, :], xt, mask=store_mask)


def test_transpose_tiled():
    M, N = 40, 24
    x = torch.arange(M * N, dtype=torch.float32).view(M, N)
    out = torch.empty(N, M, dtype=torch.float32)
    grid = (triton.cdiv(M, 16), triton.cdiv(N, 8))
    transpose_tiled_kernel[grid](x, out, M, N, BLOCK_M=16, BLOCK_N=8)
    torch.testing.assert_close(out, x.t())
    assert grid == (3, 3)
    assert grid[0] * grid[1] == 9


if HAS_GPU:
    test_transpose_tiled()
    print("tiled transpose: 3x3 programs, 40x24 -> 24x40")


In [ ]:
@triton.jit
def copy_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    tl.store(out_ptr + offs, tl.load(x_ptr + offs, mask=mask), mask=mask)


def test_num_warps():
    x = torch.arange(1000, dtype=torch.float32)
    for warps in (1, 2, 4, 8):
        out = torch.empty_like(x)
        copy_kernel[(triton.cdiv(1000, 256),)](
            x, out, 1000, BLOCK=256, num_warps=warps
        )
        torch.testing.assert_close(out, x)
    assert 256 / (32 * 1) == 8
    assert 256 / (32 * 8) == 1


if HAS_GPU:
    test_num_warps()
    print("num_warps 1,2,4,8: values unchanged; elements/thread 8,4,2,1")


### Coalesced vs uncoalesced: the byte math

- **Row read.** A $4096\times4096$ `float32` matrix has $4096\cdot4=16{,}384$ bytes per row; a warp reading 32 consecutive words touches 128 bytes, one transaction.
- **Column read.** Lane addresses are $a_\ell=a_0+4s\ell$ bytes with $s=4096$, so 32 lanes touch 32 different 128-byte lines: $Q_{\text{eff}}=32\cdot128=4096$ bytes for $Q_{\text{useful}}=32\cdot4=128$ bytes, an efficiency of $1/32$.
- **Matrix size.** $Q_{\text{data}}=4096^2\cdot4=67{,}108{,}864$ bytes $=64$ MiB.
- **Coalesced traffic.** Read plus write moves $2Q_{\text{data}}=128$ MiB.
- **One uncoalesced side.** Under the no-cache-reuse model of §12.14, it moves $32Q_{\text{data}}$; adding the other side gives $Q_{\text{bad}}=33Q_{\text{data}}=2112$ MiB, a ratio $33/2=16.5$.
- **Time at 1000 GB/s.** $t_{\text{good}}=134{,}217{,}728/10^{12}=0.134$ ms and $t_{\text{bad}}=2{,}214{,}592{,}512/10^{12}=2.215$ ms.
- **Tiled transpose.** It avoids the 32× store penalty by converting the tile in registers/shared memory and writing output rows contiguously; the conversion costs about $2Tb$ per tile in shared memory, much cheaper than a 32× global penalty.


In [ ]:
def test_traffic_model():
    n, b = 4096, 4
    q_data = n * n * b
    q_good = 2 * q_data
    q_bad = 33 * q_data
    assert 4096 * 4 == 16_384
    assert 32 * 4 == 128
    assert q_data == 67_108_864
    assert q_data // (1024 * 1024) == 64
    assert q_good == 134_217_728
    assert q_bad == 2_214_592_512
    assert q_bad // (1024 * 1024) == 2112
    assert 32 * 128 == 4096
    assert q_bad / q_good == 16.5

    bw = 10**12
    t_good_ms = q_good / bw * 1e3
    t_bad_ms = q_bad / bw * 1e3
    assert abs(t_good_ms - 0.134217728) < 1e-9
    assert abs(t_bad_ms - 2.214592512) < 1e-9

    # One 32x32 fp32 tile: write plus read through shared memory.
    tile, tb = 32 * 32, 4
    q_conv = 2 * tile * tb
    assert q_conv == 8192
    assert 1000 * q_conv == 8_192_000

    # Stride model: 32 fp32 words, 128-byte lines, no reuse.
    for stride in (1, 2, 4, 8, 16, 32, 4096):
        transactions = min(32, max(1, stride))
        moved = 128 * transactions
        useful = 32 * 4
        assert useful / moved == 1 / transactions


test_traffic_model()
print("traffic model: 64 MiB matrix, 128 MiB coalesced, 2112 MiB one-sided")


## Pitfalls

- **How to use this table.** Read the failure, then the fix; each row is a check you can apply to a layout argument.

| Pitfall | What happens | Fix |
|---|---|---|
| Assuming a fixed thread mapping | Code silently depends on a compiler choice | State the logical map; test values, not owners |
| `tl.trans` vs `torch.transpose` | You expect a stride change; Triton permutes a distributed tile | Use `tl.trans` for kernel tiles, `torch` ops for host tensors |
| Ignoring layout-conversion cost | The arithmetic is cheap but the kernel is slow | Convert once, keep producers and consumers aligned |
| Confusing warps with threads | Counts are off by 32 | `threads = 32 * num_warps` on NVIDIA |
| Tile length not a power of two | `tl.arange` or block specialization fails or pads | Pad to the next power of two and mask |
| Too many elements/thread | Register spills to local memory | Raise `num_warps` or shrink the tile |
| Too few elements/thread | Lanes idle; utilization collapses | Grow the tile or lower `num_warps` |
| Treating a slice as contiguous | Wrong strides or masks | Use parent coordinates and explicit offsets |
| Assuming same shape = same layout | A conversion appears at the elementwise join | Track provenance; convert deliberately |
| Expecting broadcasting to be free | Hidden conversion or replicated loads | Materialize once when reuse is high |


## Self-check

1. For $T=8$ and $W=1$, list the thread, lane, and register of every live index. How many lanes are idle?
2. For $T=16$ and $W=2$, what is the warp chunk $C$? Where does $(i,j)=(2,3)$ land?
3. For a 4096-element tile, what is elements/thread with 4 warps? With 8?
4. Why does the 16×16 blocked example have 8 elements per thread when $s=(2,2)$, $t=(4,8)$, $w=(1,1)$?
5. What does `tl.trans` change, and what does it leave alone?
6. Why can `tl.trans(x) + y` be slower than `x + y` even though the arithmetic is identical?
7. A warp reads 32 fp32 words with stride 8. How many 128-byte transactions, and what is the efficiency?
8. A $4096\times4096$ fp32 transpose has one uncoalesced side. What is the traffic ratio to the fully coalesced version?
9. An 8×8 parent is sliced at $(2,1)$ to $4\times4$. What fraction of the parent is live, and where does the slice corner $(3,3)$ land?


## Summary

- **Shape vs layout.** Logical shape and physical layout are different objects. Shape is yours; the concrete layout is chosen by the compiler.
- **Counts.** Elements/thread $=T/(32W)$, threads/program $=32W$, and a tile smaller than the thread count leaves lanes idle.
- **Index and transpose.** `tl.arange` manufactures logical indices. `tl.trans` changes logical axes and may require a conversion; it is not a global-memory copy by itself.
- **Layout families.** Blocked layouts are regular per-thread tiling, slice layouts are restricted windows, and distributed tensors are owned tile by tile and program by program.
- **Compatibility.** Shape plus layout; two kernels with the same FLOPs can move very different numbers of bytes.
- **Performance.** $t\ge Q_{\text{eff}}/\mathrm{BW}$; layout determines $Q_{\text{eff}}$ through coalescing and conversions.

**Where this leads.** The ten exercises below test the map, the counts, the conversion cost, and the coalescing model. Chapter 13 turns the efficiency table into alignment, vectorization, and bandwidth measurements.

**Next:** [Chapter 13 — Memory Optimization](../ch13_memory_optimization/13_complete.ipynb)


## Exercises

1. Draw the 8-element tile as a 2-warp map with 4 elements per thread. Write the thread/lane/register table and compute the utilization.
2. Compute elements/thread for 8×8, 16×16, and 32×32 tiles with 1, 2, and 4 warps. Which combination is most likely to spill, and why?
3. Extend `teaching_layout` to 1024 elements and 4 warps. Assert the inverse for every live index and report the number of active threads.
4. Modify the `tl.arange` kernel to start at 100 and length 16. Verify the output values and explain what the layout does **not** change.
5. Build a 2-D tile with rows $[2,3]$ and columns $[4,5,6]$. Write the flat offsets by hand, then check them with asserts.
6. Transpose a $5\times7$ tile using an $8\times8$ block. Write both masks and explain why the store mask is not the transpose of the load mask.
7. Apply `tl.trans` twice in one kernel. Show logically that the result is the input, then explain why it may still cost two conversions.
8. Compute the traffic for a $2048\times2048$ fp32 transpose with one uncoalesced side. Compare with the $4096\times4096$ case.
9. Design a copy kernel whose warp reads with stride 4. Predict transactions and efficiency, then propose a tiled alternative.
10. Give a kernel where two tensors have the same shape but different layouts, and explain where the compiler inserts the conversion.


## Run everything

- **CPU tests.** Run directly; they cover the worked tables, host map, inverse, layout arithmetic, and traffic model.
- **GPU tests.** Collected in one loop; without a GPU each one prints `SKIP <name>: ...` and the notebook continues.


In [ ]:
CPU_TESTS = [
    test_worked_tables,
    test_host_layout_map,
    test_inverse_layout,
    test_layout_tables,
    test_traffic_model,
]

GPU_TESTS = [
    test_arange_kernel,
    test_tile2d_kernel,
    test_trans_kernel,
    test_transpose_tiled,
    test_num_warps,
]

print("CPU tests")
for test in CPU_TESTS:
    test()

print()
print("GPU tests")
for test in GPU_TESTS:
    run_if_gpu(test)

print()
print("done: CPU checks passed; GPU checks ran or printed SKIP")
